# LeetCode 2208 — Minimum Operations to Halve Array Sum bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-halve-array-sum/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def halveArray(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhìn mỗi thao tác "giảm một nửa" như việc **thu được một lượng giảm** `x/2`, và nhận ra mỗi số `v` cung cấp một **chuỗi lượng giảm giảm dần** `v/2, v/4, v/8, ...`.
2. Chứng minh tham lam **"luôn giảm một nửa số lớn nhất hiện tại"** bằng lập luận trao đổi: `k` thao tác tham lam cho **tổng lượng giảm lớn nhất** trong mọi cách dùng `k` thao tác.
3. Cài tham lam bằng **max-heap** (`heapq` với đổi dấu) và bằng **SortedList**, so với cách tìm max tuyến tính mỗi bước.
4. Hiểu vì sao phép chia đôi số thực **chính xác** trong `float` (với miền giá trị của đề) và cách so sánh an toàn bằng số nguyên (nhân đôi hai vế).
5. Kiểm chứng tham lam bằng **tìm kiếm vét cạn** (iterative deepening, số học phân số chính xác) trên dữ liệu nhỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums`. Mỗi thao tác: chọn **một số bất kỳ** trong mảng và thay nó bằng **đúng một nửa** của nó (có thể là số thực; số đã giảm vẫn có thể được chọn lại).

Trả về số thao tác **ít nhất** để **tổng** của mảng giảm đi **ít nhất một nửa** so với tổng ban đầu.

**Ví dụ 1:** `nums = [5,19,8,1]` → `3`. Tổng 33, cần giảm ≥ 16,5: `19 → 9,5`, `9,5 → 4,75`, `8 → 4` ⇒ giảm `9,5 + 4,75 + 4 = 18,25`.

**Ví dụ 2:** `nums = [3,8,20]` → `3`. Tổng 31, cần giảm ≥ 15,5: `20 → 10`, `10 → 5`, `8 → 4` (hoặc `3 → 1,5`) ⇒ đủ.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ nums[i] ≤ 10⁷`.

In [1]:
import random
import time
import heapq
from fractions import Fraction
from itertools import product
from typing import List
from sortedcontainers import SortedList

examples = [([5, 19, 8, 1], 3), ([3, 8, 20], 3)]
for nums, exp in examples:
    print(nums, "tổng", sum(nums), "cần giảm >=", sum(nums) / 2, "-> kỳ vọng", exp)

[5, 19, 8, 1] tổng 33 cần giảm >= 16.5 -> kỳ vọng 3
[3, 8, 20] tổng 31 cần giảm >= 15.5 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 `heapq` là min-heap — đổi dấu để có max-heap

`heapq.heappop` lấy phần tử **nhỏ nhất**. Muốn lấy lớn nhất: lưu `−x`. `heapq.heapreplace(h, y)` = pop rồi push trong **một** thao tác (nhanh hơn hai lệnh riêng).

In [2]:
h = [-x for x in [5, 19, 8, 1]]
heapq.heapify(h)
top = -h[0]
heapq.heapreplace(h, -top / 2)            # thay đỉnh (19) bằng 9.5
print("lớn nhất ban đầu:", top, "| heap sau khi thay:", sorted(-x for x in h))

lớn nhất ban đầu: 19 | heap sau khi thay: [1, 5, 8, 9.5]


### 3.2 Chia đôi số thực là phép toán **chính xác** trong `float`

`float` (IEEE-754 double) lưu số dạng `m · 2^e`. Chia đôi chỉ giảm `e` đi 1, **không làm tròn** — miễn là không xuống dưới vùng số rất nhỏ (`~10⁻³⁰⁸`). Với `nums[i] ≤ 10⁷` và số thao tác không quá vài triệu lần trên mỗi số, các giá trị và **tổng các lượng giảm** vẫn có thể bị làm tròn khi **cộng** nhiều số khác bậc; nhưng so sánh với ngưỡng có biên an toàn rất lớn. Ta sẽ kiểm bằng `Fraction` (chính xác tuyệt đối).

In [3]:
x = 19.0
for _ in range(5):
    x /= 2
print(x, "==", Fraction(19, 32), "?", Fraction(x) == Fraction(19, 32))
print("0.1 + 0.2 == 0.3?", 0.1 + 0.2 == 0.3, "(phép CỘNG có thể làm tròn, phép CHIA ĐÔI thì không)")

0.59375 == 19/32 ? True
0.1 + 0.2 == 0.3? False (phép CỘNG có thể làm tròn, phép CHIA ĐÔI thì không)


### 3.3 So sánh "ít nhất một nửa" bằng số nguyên

`reduced ≥ total/2` ⇔ `2·reduced ≥ total`. Nếu muốn tránh hẳn số thực, có thể **nhân mọi giá trị với `2^K`** cho đủ lớn (`K` = số lần chia tối đa trên một số) để mọi phép chia đôi đều là số nguyên — ta dùng ý này trong Cách 5.

In [4]:
total = 33
for reduced in (Fraction(33, 2), Fraction(16), Fraction(73, 4)):
    print(f"reduced={reduced}: reduced >= total/2? {reduced >= Fraction(total, 2)} | 2*reduced >= total? {2 * reduced >= total}")

reduced=33/2: reduced >= total/2? True | 2*reduced >= total? True
reduced=16: reduced >= total/2? False | 2*reduced >= total? False
reduced=73/4: reduced >= total/2? True | 2*reduced >= total? True


### 3.4 `SortedList` (thư viện `sortedcontainers`)

Danh sách luôn được sắp: `add(x)` và `pop()` (lấy phần tử lớn nhất ở cuối) đều `O(log n)`. Là phương án thay thế heap khi cần cả min lẫn max.

In [5]:
sl = SortedList([5, 19, 8, 1])
big = sl.pop()          # lớn nhất
sl.add(big / 2)
print("lấy", big, "| còn:", list(sl))

lấy 19 | còn: [1, 5, 8, 9.5]


## 4. Ý tưởng nền tảng / trực giác

**Mỗi thao tác "mua" một lượng giảm.** Chia đôi số `x` làm tổng giảm `x/2`. Muốn đạt ngưỡng `total/2` bằng ít thao tác nhất ⇒ mỗi thao tác nên giảm **càng nhiều càng tốt** ⇒ chọn số **lớn nhất hiện tại**.

Tính tay ví dụ 1 (tổng 33, cần 16,5):

| Bước | Mảng trước | Chọn | Giảm thêm | Tổng đã giảm |
|---|---|---|---|---|
| 1 | 5, 19, 8, 1 | 19 | 9,5 | 9,5 |
| 2 | 5, **9,5**, 8, 1 | 9,5 | 4,75 | 14,25 |
| 3 | 5, 4,75, **8**, 1 | 8 | 4 | 18,25 ≥ 16,5 ✓ |

Chú ý bước 2: sau khi 19 thành 9,5, số lớn nhất vẫn là 9,5 (> 8) ⇒ phải **chọn lại** số vừa giảm. Vì vậy cần cấu trúc dữ liệu cho phép lấy max **sau mỗi lần cập nhật** — heap.

**Góc nhìn "chuỗi lượng giảm":** số `v` cung cấp các lượng giảm `v/2, v/4, v/8, ...` và phải dùng **theo thứ tự** (muốn giảm `v/4` thì đã giảm `v/2` trước). Vì chuỗi giảm dần, `k` lượng giảm lớn nhất trong **tất cả** các chuỗi tự động là các tiền tố của từng chuỗi ⇒ hợp lệ ⇒ tham lam lấy `k` lượng lớn nhất là tối ưu.

In [6]:
nums = [5, 19, 8, 1]
chains = {v: [v / 2 ** t for t in range(1, 5)] for v in nums}
for v, ch in chains.items():
    print(f"số {v:2d} cung cấp các lượng giảm {ch}")
all_reductions = sorted((r for ch in chains.values() for r in ch), reverse=True)
print("các lượng giảm lớn nhất:", all_reductions[:5], "| cộng dồn:", [sum(all_reductions[:k]) for k in range(1, 5)])

số  5 cung cấp các lượng giảm [2.5, 1.25, 0.625, 0.3125]
số 19 cung cấp các lượng giảm [9.5, 4.75, 2.375, 1.1875]
số  8 cung cấp các lượng giảm [4.0, 2.0, 1.0, 0.5]
số  1 cung cấp các lượng giảm [0.5, 0.25, 0.125, 0.0625]
các lượng giảm lớn nhất: [9.5, 4.75, 4.0, 2.5, 2.375] | cộng dồn: [9.5, 14.25, 18.25, 20.75]


## 5. Cách 1 — Brute Force / baseline: tìm kiếm vét cạn theo độ sâu tăng dần

**Ý tưởng (iterative deepening):** với `d = 1, 2, 3, ...`, thử mọi dãy `d` lựa chọn chỉ số (`n^d` dãy), mô phỏng bằng `Fraction` (chính xác). Độ sâu `d` nhỏ nhất có dãy đạt ngưỡng chính là đáp án. Không dựa vào bất kỳ giả định tham lam nào.

In [7]:
def halve_brute(nums, max_depth=8):
    total = Fraction(sum(nums))
    start = tuple(Fraction(v) for v in nums)
    def dfs(arr, reduced, d):
        if 2 * reduced >= total:
            return True
        if d == 0:
            return False
        tried = set()
        for i, v in enumerate(arr):
            if v in tried:            # chọn hai số bằng nhau cho kết quả như nhau
                continue
            tried.add(v)
            new = arr[:i] + (v / 2,) + arr[i + 1:]
            if dfs(new, reduced + v / 2, d - 1):
                return True
        return False
    for d in range(0, max_depth + 1):
        if dfs(start, Fraction(0), d):
            return d
    return None

for nums, exp in examples:
    got = halve_brute(nums); print(nums, "->", got); assert got == exp

[5, 19, 8, 1] -> 3
[3, 8, 20] -> 3


**Độ phức tạp:** tới `n^d` dãy ở độ sâu `d` — hàm mũ. Chỉ dùng cho `n ≤ 5`, đáp án ≤ ~7. Space `O(d)` cho đệ quy.

**Nút thắt:** ta thử mọi lựa chọn trong khi chỉ có **một** lựa chọn tốt nhất mỗi bước (số lớn nhất) — cần chứng minh và dùng cấu trúc lấy max nhanh.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam:** lặp lại "lấy số lớn nhất `x`, thay bằng `x/2`, cộng `x/2` vào tổng đã giảm" cho tới khi tổng đã giảm `≥ total/2`.

**Bổ đề trao đổi.** Gọi `R_k` = tổng lượng giảm của `k` thao tác tham lam, `R*_k` = tổng lượng giảm lớn nhất có thể với `k` thao tác. Khi đó `R_k = R*_k`.
- Mọi cách dùng `k` thao tác tương ứng với việc chọn `k` phần tử từ **hợp các chuỗi** `{v/2, v/4, ...}` sao cho trong mỗi chuỗi ta chọn một **tiền tố**.
- Tổng lớn nhất khi chọn `k` phần tử **bất kỳ** (bỏ ràng buộc tiền tố) là tổng `k` phần tử lớn nhất. Vì mỗi chuỗi **giảm dần**, `k` phần tử lớn nhất tự động lập thành tiền tố của từng chuỗi ⇒ ràng buộc được thỏa ⇒ đó cũng là tối ưu có ràng buộc.
- Tham lam (lấy max hiện tại) chính là lấy lần lượt các phần tử lớn nhất của hợp chuỗi.

**Hệ quả:** số thao tác nhỏ nhất = `k` nhỏ nhất với `R_k ≥ total/2` — đúng là điểm dừng của tham lam.

**Quy tắc "nếu… thì…":** **nếu** tổng đã giảm `< total/2` **thì** chia đôi số lớn nhất; **ngược lại** dừng.

**Chặn trên số thao tác:** chia đôi **mỗi** số đúng một lần đã giảm đúng `total/2`, nên đáp án luôn `≤ n` (chứng minh chi tiết ở mục 11).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam, tìm max tuyến tính mỗi bước

Đúng ý tham lam, nhưng mỗi bước `max()` tốn `O(n)` ⇒ `O(n · ops)`.

In [8]:
def halve_linear(nums):
    arr = [float(v) for v in nums]
    target = sum(arr) / 2
    reduced, ops = 0.0, 0
    while reduced < target:
        i = max(range(len(arr)), key=arr.__getitem__)
        arr[i] /= 2
        reduced += arr[i]
        ops += 1
    return ops

for nums, exp in examples:
    assert halve_linear(nums) == exp
print([halve_linear(nums) for nums, _ in examples])

[3, 3]


**Độ phức tạp Cách 2:** `O(n · ops)` — với `n = 10⁵` và `ops` cỡ `n` thì `10¹⁰`, TLE.

### 7.2 Cách 3 — Max-heap (bản nộp chính)

**Giải thích từng đoạn:**
1. `h = [−v for v in nums]`, `heapify(h)` trong `O(n)`.
2. `target = sum(nums) / 2`, `reduced = 0`.
3. Trong khi `reduced < target`: `x = −h[0]`, `heapreplace(h, −x/2)`, `reduced += x/2`, `ops += 1`.

In [9]:
def halve_heap(nums):
    h = [-float(v) for v in nums]
    heapq.heapify(h)
    target = sum(nums) / 2
    reduced, ops = 0.0, 0
    while reduced < target:
        x = -h[0]
        heapq.heapreplace(h, -x / 2)
        reduced += x / 2
        ops += 1
    return ops

for nums, exp in examples:
    assert halve_heap(nums) == exp
print([halve_heap(nums) for nums, _ in examples])

[3, 3]


**Độ phức tạp Cách 3:** `heapify` `O(n)`, mỗi thao tác `O(log n)` ⇒ `O(n + ops · log n)`. Space `O(n)`.

### 7.3 Cách 4 — `SortedList`

Thay heap bằng danh sách có thứ tự; `pop()` lấy lớn nhất, `add` chèn nửa.

In [10]:
def halve_sortedlist(nums):
    sl = SortedList(float(v) for v in nums)
    target = sum(nums) / 2
    reduced, ops = 0.0, 0
    while reduced < target:
        x = sl.pop()
        sl.add(x / 2)
        reduced += x / 2
        ops += 1
    return ops

for nums, exp in examples:
    assert halve_sortedlist(nums) == exp
print([halve_sortedlist(nums) for nums, _ in examples])

[3, 3]


**Độ phức tạp Cách 4:** `O(n log n + ops · log n)`. Hằng số lớn hơn heap.

### 7.4 Cách 5 — Heap với số nguyên đã "phóng to" (không dùng số thực)

Nhân mọi số với `2^K` (`K = 25` là đủ an toàn ở đây; xem mục 11 về số lần chia tối đa trên một số). Khi đó chia đôi luôn là **số nguyên** (cho tới `K` lần trên mỗi số) và mọi so sánh là chính xác tuyệt đối. Python có số nguyên lớn nên không lo tràn.

In [11]:
def halve_heap_int(nums, K=25):
    h = [-(v << K) for v in nums]
    heapq.heapify(h)
    total = sum(nums) << K
    reduced, ops = 0, 0
    while 2 * reduced < total:
        x = -h[0]
        assert x % 2 == 0, "K chưa đủ lớn"
        heapq.heapreplace(h, -(x // 2))
        reduced += x // 2
        ops += 1
    return ops

for nums, exp in examples:
    assert halve_heap_int(nums) == exp
print([halve_heap_int(nums) for nums, _ in examples])

[3, 3]


**Độ phức tạp Cách 5:** như Cách 3; số nguyên lớn (≈ 50 bit) vẫn nhanh.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 `[3, 8, 20]` (tổng 31, cần giảm ≥ 15,5):

| Bước | Max lấy ra | Thay bằng | reduced | Heap sau (giá trị) |
|---|---|---|---|---|
| 1 | 20 | 10 | 10 | 10, 8, 3 |
| 2 | 10 | 5 | 15 | 8, 5, 3 |
| 3 | 8 | 4 | 19 ≥ 15,5 ✓ | 5, 4, 3 |

Đáp án `3`. (Đề minh họa chọn `3` ở bước 3; chọn `8` giảm nhiều hơn nhưng cùng số bước.)

In [12]:
def trace_heap(nums):
    h = [-float(v) for v in nums]; heapq.heapify(h)
    target = sum(nums) / 2; reduced, ops = 0.0, 0
    print(f"tổng = {sum(nums)}, cần giảm >= {target}")
    while reduced < target:
        x = -h[0]; heapq.heapreplace(h, -x / 2); reduced += x / 2; ops += 1
        print(f"bước {ops}: chia đôi {x} -> {x / 2}, reduced = {reduced}, mảng = {sorted((-v for v in h), reverse=True)}")
    return ops

print("Đáp án:", trace_heap([3, 8, 20]))
print()
print("Đáp án:", trace_heap([5, 19, 8, 1]))

tổng = 31, cần giảm >= 15.5
bước 1: chia đôi 20.0 -> 10.0, reduced = 10.0, mảng = [10.0, 8.0, 3.0]
bước 2: chia đôi 10.0 -> 5.0, reduced = 15.0, mảng = [8.0, 5.0, 3.0]
bước 3: chia đôi 8.0 -> 4.0, reduced = 19.0, mảng = [5.0, 4.0, 3.0]
Đáp án: 3

tổng = 33, cần giảm >= 16.5
bước 1: chia đôi 19.0 -> 9.5, reduced = 9.5, mảng = [9.5, 8.0, 5.0, 1.0]
bước 2: chia đôi 9.5 -> 4.75, reduced = 14.25, mảng = [8.0, 5.0, 4.75, 1.0]
bước 3: chia đôi 8.0 -> 4.0, reduced = 18.25, mảng = [5.0, 4.75, 4.0, 1.0]
Đáp án: 3


In [13]:
# Mảng có một số rất lớn: số đó bị chọn lại nhiều lần liên tiếp
print("Đáp án:", trace_heap([1, 1, 100]))

tổng = 102, cần giảm >= 51.0
bước 1: chia đôi 100.0 -> 50.0, reduced = 50.0, mảng = [50.0, 1.0, 1.0]
bước 2: chia đôi 50.0 -> 25.0, reduced = 75.0, mảng = [25.0, 1.0, 1.0]
Đáp án: 2


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def halveArray(self, nums: List[int]) -> int:
        h = [-float(v) for v in nums]        # max-heap bằng đổi dấu
        heapq.heapify(h)
        target = sum(nums) / 2
        reduced, ops = 0.0, 0
        while reduced < target:
            x = -h[0]                        # số lớn nhất hiện tại
            heapq.heapreplace(h, -x / 2)     # thay bằng một nửa
            reduced += x / 2
            ops += 1
        return ops

for nums, exp in examples:
    assert Solution().halveArray(nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Mô hình.** Một chiến lược dùng `k` thao tác chia đôi số thứ `i` tổng cộng `t_i` lần (`Σ t_i = k`); thứ tự các thao tác không ảnh hưởng tổng cuối. Lượng giảm từ số `v_i` là `v_i/2 + v_i/4 + ... + v_i/2^{t_i}` — tức là tổng `t_i` phần tử **đầu** của chuỗi giảm dần `C_i = (v_i/2, v_i/4, ...)`.

**Tối ưu cho mỗi `k`.** Tổng lượng giảm của mọi chiến lược `k` thao tác ≤ tổng `k` phần tử lớn nhất của đa tập `⋃ C_i` (bỏ ràng buộc tiền tố chỉ làm tăng giá trị tối ưu). Gọi `S` là `k` phần tử lớn nhất đó (phá hòa tùy ý nhưng ưu tiên phần tử sớm hơn trong cùng chuỗi). Vì mỗi `C_i` giảm nghiêm ngặt, nếu `S` chứa `v_i/2^t` thì cũng chứa mọi `v_i/2^s` với `s < t` ⇒ `S` lập thành các tiền tố ⇒ đạt được. Tham lam (lấy max hiện tại của heap — chính là phần tử lớn nhất chưa dùng ở đầu mỗi chuỗi) chọn đúng `S`.

**Dừng.** Vì `R_k` (lượng giảm tham lam) là tối ưu cho mọi `k`, số thao tác tối thiểu là `k` nhỏ nhất có `R_k ≥ total/2` — đúng điều kiện dừng. Vòng lặp luôn dừng vì chia đôi mọi số `t` lần đủ lớn cho lượng giảm tiến tới `total`.

**Độ chính xác số thực.** Mọi giá trị trong heap có dạng `v/2^t` — biểu diễn chính xác bằng `float` khi `t` không quá lớn. Chỉ phép **cộng** dồn `reduced` có thể làm tròn; ta kiểm bằng Cách 5 (số nguyên chính xác) trên dữ liệu lớn ở mục 12.

In [15]:
# Kiểm chứng: R_k tham lam == tổng k phần tử lớn nhất của hợp các chuỗi (dùng Fraction chính xác)
rng = random.Random(10)
for _ in range(200):
    nums = [rng.randint(1, 50) for _ in range(rng.randint(1, 6))]
    K = 12
    pool = sorted((Fraction(v, 2 ** t) for v in nums for t in range(1, K + 1)), reverse=True)
    h = [-Fraction(v) for v in nums]; heapq.heapify(h); R = Fraction(0)
    for k in range(1, K + 1):
        x = -h[0]; heapq.heapreplace(h, -x / 2); R += x / 2
        assert R == sum(pool[:k])
print("R_k tham lam bằng tổng k lượng giảm lớn nhất (k <= 12) trên 200 mảng")

R_k tham lam bằng tổng k lượng giảm lớn nhất (k <= 12) trên 200 mảng


## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `ops` = đáp án.

**Chặn trên của `ops`:** chia đôi **mỗi** số một lần đã giảm đúng `total/2` ⇒ `ops ≤ n`. Vậy tham lam heap chạy `O(n log n)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn IDDFS | `O(n^ops)` | `O(ops)` |
| 2. Max tuyến tính | `O(n · ops)` = `O(n²)` | `O(n)` |
| 3. Max-heap | `O(n + ops log n)` = `O(n log n)` | `O(n)` |
| 4. SortedList | `O(n log n)` | `O(n)` |
| 5. Heap số nguyên | `O(n log n)` | `O(n)` |

Vì `ops ≤ n`, mỗi số bị chia tối đa `ops ≤ 10⁵` lần về lý thuyết; thực tế số bị chia nhiều nhất là số lớn nhất, và nó chỉ bị chia khoảng `log₂(n · max/min)` lần trước khi nhỏ hơn các số khác — `K = 25` ở Cách 5 có `assert` để phát hiện nếu không đủ.

In [16]:
rng = random.Random(2208)
big = [rng.randint(1, 10**7) for _ in range(10**5)]
skew = [10**7] + [1] * (10**5 - 1)
for name, fn in [("heap", halve_heap), ("sortedlist", halve_sortedlist), ("heap_int", halve_heap_int),
                 ("Solution", lambda a: Solution().halveArray(a))]:
    t = time.perf_counter(); r1 = fn(big); r2 = fn(skew)
    print(f"{name:10s}: ngẫu nhiên={r1}, lệch={r2} ({time.perf_counter() - t:.3f}s)")
print("ops <= n luôn đúng:", halve_heap(big) <= len(big))

heap      : ngẫu nhiên=77501, lệch=2 (0.048s)


sortedlist: ngẫu nhiên=77501, lệch=2 (0.079s)


heap_int  : ngẫu nhiên=77501, lệch=2 (0.072s)
Solution  : ngẫu nhiên=77501, lệch=2 (0.047s)


ops <= n luôn đúng: True


**Một số bị chia tối đa bao nhiêu lần?** Đây là câu hỏi quyết định `K` ở Cách 5 và độ an toàn của `float`. Số lớn nhất `M` chỉ bị chọn lại khi nó vẫn còn lớn nhất; sau `t` lần nó còn `M/2^t`. Một chặn trên thô là `log₂(M) ≤ 24` lần (sau đó `M` nhỏ hơn 1). Nhưng thực tế còn nhỏ hơn nhiều: nếu một phần tử `x` sắp bị chia lần thứ ba thì `x/4` đang là số lớn nhất, nên tổng hiện tại `≤ (số phần tử) · x/4`… và hai lần chia trước đã giảm `3x/4`, thường đã vượt nửa tổng. Ô dưới đếm số lần chia lớn nhất trên một phần tử ở vài dữ liệu "xấu" — kết quả rất nhỏ, nên `K = 25` ở Cách 5 dư sức an toàn (và có `assert` canh chừng).

In [17]:
def max_halvings_per_element(nums):
    h = [(-float(v), i) for i, v in enumerate(nums)]; heapq.heapify(h)
    target = sum(nums) / 2; reduced = 0.0; times = [0] * len(nums)
    while reduced < target:
        x, i = h[0]; x = -x
        heapq.heapreplace(h, (-x / 2, i)); reduced += x / 2; times[i] += 1
    return max(times)

cases = {
    "1 số 10^7 + 10^5 số 1": [10**7] + [1] * 10**5,
    "1 số 10^7 + 10 số 1": [10**7] + [1] * 10,
    "lũy thừa 2 giảm dần": [2 ** k for k in range(23, 0, -1)],
    "ngẫu nhiên": [random.Random(1).randint(1, 10**7) for _ in range(10**4)],
}
for name, arr in cases.items():
    print(f"{name:24s}: một phần tử bị chia tối đa {max_halvings_per_element(arr)} lần")

1 số 10^7 + 10^5 số 1   : một phần tử bị chia tối đa 2 lần
1 số 10^7 + 10 số 1     : một phần tử bị chia tối đa 2 lần
lũy thừa 2 giảm dần     : một phần tử bị chia tối đa 2 lần
ngẫu nhiên              : một phần tử bị chia tối đa 1 lần


## 12. Bộ test

- Ví dụ đề.
- Biên: một phần tử (luôn 1 thao tác); mọi phần tử bằng nhau (cần đúng `n` thao tác — mỗi số một lần); một số rất lớn với nhiều số nhỏ; giá trị lớn nhất `10⁷`.
- Đối chiếu tham lam với **vét cạn chính xác** (Fraction) trên mảng nhỏ ngẫu nhiên (seed cố định).
- Đối chiếu Cách 3 (float) với Cách 5 (số nguyên chính xác) trên mảng lớn.

In [18]:
fns = {"brute": halve_brute, "linear": halve_linear, "heap": halve_heap, "sortedlist": halve_sortedlist,
       "heap_int": halve_heap_int, "Solution": lambda a: Solution().halveArray(a)}
edge = [([1], 1), ([10**7], 1), ([4, 4, 4, 4], 4), ([6, 6, 6], 3), ([1, 1, 100], 2), ([1, 1, 1, 1, 1], 5), ([1, 2, 4, 8], 3)]
for nums, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums, fn(nums))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [19]:
rng = random.Random(42)
for _ in range(150):
    nums = [rng.randint(1, rng.choice([5, 20, 100])) for _ in range(rng.randint(1, 4))]
    want = halve_brute(nums)
    for name, fn in fns.items():
        assert fn(nums) == want, (name, nums)
for _ in range(20):
    nums = [rng.randint(1, 10**7) for _ in range(rng.randint(1, 3000))]
    assert halve_heap(nums) == halve_heap_int(nums) == halve_sortedlist(nums)
print("150 test nhỏ khớp vét cạn chính xác; 20 test lớn: float khớp số nguyên chính xác")

150 test nhỏ khớp vét cạn chính xác; 20 test lớn: float khớp số nguyên chính xác


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng chia nguyên `x // 2`

Đề cho phép số thực; `x // 2` làm **mất phần lẻ**, khiến tổng giảm **nhiều hơn** thực tế ⇒ có thể dừng sớm (đáp án nhỏ hơn đúng). Tìm phản ví dụ bằng máy:

In [20]:
def wrong_floor_div(nums):
    h = [-v for v in nums]; heapq.heapify(h)
    target = sum(nums) / 2; reduced = ops = 0
    while reduced < target:
        x = -h[0]
        heapq.heapreplace(h, -(x // 2))
        reduced += x - x // 2                       # SAI: giảm thật là x/2, không phải x - x//2
        ops += 1
    return ops

bad = next(list(t) for n in range(1, 4) for t in product(range(1, 8), repeat=n) if wrong_floor_div(list(t)) != halve_heap(list(t)))
print("phản ví dụ:", bad, "| sai:", wrong_floor_div(bad), "| đúng:", halve_heap(bad))

phản ví dụ: [1, 1] | sai: 1 | đúng: 2


In [21]:
print("sửa (chia số thực):", Solution().halveArray(bad))

sửa (chia số thực): 2


### Lỗi 2 — Dùng min-heap mà quên đổi dấu (chia đôi số **nhỏ** nhất)

Tệ hơn cả "chậm": liên tục chia đôi số nhỏ nhất chỉ giảm được tổng cộng tối đa bằng chính số đó (chuỗi `x/2 + x/4 + ... < x`), nên nếu số nhỏ nhất `< total/2` thì vòng lặp **không bao giờ dừng**. Ta thêm "cầu chì" 1000 bước để quan sát.

In [22]:
def wrong_min_heap(nums):
    h = [float(v) for v in nums]; heapq.heapify(h)
    target = sum(nums) / 2; reduced, ops = 0.0, 0
    while reduced < target and ops < 1000:           # "cầu chì": bản sai có thể lặp vô hạn
        x = h[0]                                     # SAI: đây là số NHỎ nhất
        heapq.heapreplace(h, x / 2); reduced += x / 2; ops += 1
    return ops if reduced >= target else f"sau {ops} bước mới giảm {reduced:.4f} < {target} (sẽ lặp mãi)"

print("sai:", wrong_min_heap([5, 19, 8, 1]), "| đúng:", 3)

sai: sau 1000 bước mới giảm 1.0000 < 16.5 (sẽ lặp mãi) | đúng: 3


In [23]:
print("sửa (đổi dấu):", halve_heap([5, 19, 8, 1]))

sửa (đổi dấu): 3


### Lỗi 3 — Sắp xếp một lần rồi chia đôi mỗi số **đúng một lần** từ lớn tới nhỏ

Bỏ qua khả năng chọn **lại** số vừa giảm (ví dụ 1 cần chia 19 hai lần).

In [24]:
def wrong_once_each(nums):
    target = sum(nums) / 2; reduced = 0.0; ops = 0
    for v in sorted(nums, reverse=True):             # SAI: mỗi số tối đa một lần
        if reduced >= target: break
        reduced += v / 2; ops += 1
    return ops

print("sai:", wrong_once_each([1, 1, 100]), "| đúng:", halve_heap([1, 1, 100]))
print("ví dụ 1:", wrong_once_each([5, 19, 8, 1]), "vs", halve_heap([5, 19, 8, 1]))

sai: 3 | đúng: 2
ví dụ 1: 4 vs 3


In [25]:
print("sửa (đưa nửa trở lại heap):", Solution().halveArray([5, 19, 8, 1]))

sửa (đưa nửa trở lại heap): 3


### Lỗi 4 — Ngưỡng sai: so với `sum(nums) // 2` (chia nguyên ngưỡng)

Với tổng lẻ, `33 // 2 = 16 < 16,5` ⇒ có thể dừng khi mới giảm `16` — chưa đủ "ít nhất một nửa".

In [26]:
def wrong_threshold(nums):
    h = [-float(v) for v in nums]; heapq.heapify(h)
    target = sum(nums) // 2                          # SAI: phải là sum / 2
    reduced, ops = 0.0, 0
    while reduced < target:
        x = -h[0]; heapq.heapreplace(h, -x / 2); reduced += x / 2; ops += 1
    return ops

bad = next(list(t) for n in range(1, 4) for t in product(range(1, 10), repeat=n) if wrong_threshold(list(t)) != halve_heap(list(t)))
print("phản ví dụ:", bad, "| sai:", wrong_threshold(bad), "| đúng:", halve_heap(bad))

phản ví dụ: [1] | sai: 0 | đúng: 1


In [27]:
print("sửa:", Solution().halveArray(bad))

sửa: 1


### Lỗi 5 — Pop rồi quên push lại phần một nửa

Dùng `heappop` mà không `heappush(−x/2)` thì số đó "biến mất" ⇒ mảng mất phần tử, tổng giảm sai và có thể hết phần tử.

In [28]:
def wrong_no_push(nums):
    h = [-float(v) for v in nums]; heapq.heapify(h)
    target = sum(nums) / 2; reduced, ops = 0.0, 0
    while reduced < target and h:
        x = -heapq.heappop(h)                        # SAI: không đưa x/2 trở lại
        reduced += x / 2; ops += 1
    return ops if reduced >= target else f"hết phần tử sau {ops} bước, mới giảm {reduced}"

print("sai:", wrong_no_push([1, 1, 100]), "| ví dụ 1:", wrong_no_push([5, 19, 8, 1]), "| đúng:", halve_heap([5, 19, 8, 1]))

sai: 3 | ví dụ 1: 4 | đúng: 3


In [29]:
print("sửa (heapreplace):", halve_heap([5, 19, 8, 1]))

sửa (heapreplace): 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Lặp lại thao tác trên phần tử lớn nhất/nhỏ nhất, phần tử thay đổi rồi quay lại tập"** ⇒ **heap** (max-heap bằng đổi dấu).
- Chứng minh tham lam bằng **"chuỗi lợi ích giảm dần"**: nếu lợi ích của lần chọn thứ `t` trên cùng một phần tử giảm dần, thì lấy `k` lợi ích lớn nhất là hợp lệ và tối ưu.
- `heapreplace` gọn và nhanh hơn `heappop` + `heappush`.
- Bài có số thực: cân nhắc **so sánh bằng số nguyên** (nhân đôi / phóng to) nếu lo sai số.
- Chặn trên đáp án (`ops ≤ n`) giúp ước lượng độ phức tạp.

## 15. Biến thể của bài

1. **LeetCode 1962 — Remove Stones to Minimize the Total:** mỗi thao tác bỏ `⌊x/2⌋` viên của đống lớn nhất, đúng `k` thao tác — cùng max-heap.
2. **LeetCode 2530 — Maximal Score After Applying K Operations:** lấy max, cộng điểm, thay bằng `⌈x/3⌉`.
3. **Giảm tổng xuống còn tối đa một phần `p`** (thay vì một nửa): đổi ngưỡng thành `(1 − p)·total`.
4. **Mỗi số chỉ được chia tối đa một lần** (bài tập 5): sắp giảm dần, lấy tiền tố.
5. **Chia cho 3 thay vì 2:** lượng giảm `2x/3`, chuỗi `2v/3, 2v/9, ...` vẫn giảm dần ⇒ tham lam vẫn đúng.

Minh họa biến thể 5 (tự dựng, kiểm bằng vét cạn chính xác):

In [30]:
def third_heap(nums):
    h = [-Fraction(v) for v in nums]; heapq.heapify(h)
    total = sum(nums); reduced = Fraction(0); ops = 0
    while 2 * reduced < total:
        x = -h[0]; heapq.heapreplace(h, -x / 3); reduced += x - x / 3; ops += 1
    return ops

def third_brute(nums, max_depth=7):
    total = Fraction(sum(nums))
    def dfs(arr, red, d):
        if 2 * red >= total: return True
        if d == 0: return False
        return any(dfs(arr[:i] + (v / 3,) + arr[i + 1:], red + v - v / 3, d - 1) for i, v in enumerate(arr))
    for d in range(max_depth + 1):
        if dfs(tuple(Fraction(v) for v in nums), Fraction(0), d): return d

rng = random.Random(3)
for _ in range(100):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 3))]
    assert third_heap(nums) == third_brute(nums)
print("Biến thể chia 3 khớp vét cạn; [5,19,8,1] ->", third_heap([5, 19, 8, 1]))

Biến thể chia 3 khớp vét cạn; [5,19,8,1] -> 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn IDDFS | thử mọi dãy lựa chọn | hàm mũ | `O(ops)` | kiểm chứng tham lam |
| 2. Max tuyến tính | tham lam, `max()` mỗi bước | `O(n²)` | `O(n)` | `n` nhỏ |
| 3. Max-heap | tham lam + `heapreplace` | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. SortedList | tham lam, danh sách có thứ tự | `O(n log n)` | `O(n)` | khi cần cả min/max |
| 5. Heap số nguyên | phóng to `2^K`, chính xác | `O(n log n)` | `O(n)` | lo sai số thực |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `halveArray([1, 1, 1, 1])` và `halveArray([100])`.
2. **Tính tay:** trace heap cho `[2, 3, 10]`.
3. **Điền TODO:** `ops_upper_bound(nums)` — chứng minh và trả về một chặn trên đơn giản của đáp án (gợi ý: `n`).
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `halve_once_each(nums)` — mỗi số được chia đôi **tối đa một lần**; trả về số thao tác ít nhất (luôn tồn tại vì chia hết mọi số giảm đúng một nửa). Kiểm bằng vét cạn mọi tập con.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def ops_upper_bound(nums):
    # TODO
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_halve_array(nums):
    # TODO
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def halve_once_each(nums):
    # TODO: sắp giảm dần, cộng v/2 tới khi đủ
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `[1,1,1,1]`: tổng 4, cần giảm 2; mỗi thao tác giảm 0,5 (luôn chọn một số 1 chưa chia). (2) Cần đúng 4 thao tác. (3) `[100]`: một thao tác giảm 50 = một nửa ⇒ 1.

**Bài 2.** (1) Tổng 15, cần 7,5. (2) `10 → 5` (5), `5 → 2,5` hay `3 → 1,5`? max là 5 ⇒ `5 → 2,5` (7,5) ⇒ đủ. (3) Đáp án 2.

**Bài 3.** (1) Chia đôi mỗi số một lần giảm `Σ v/2 = total/2`. (2) Vậy `n` thao tác luôn đủ. (3) Trả `len(nums)`.

**Bài 4.** (1) Max-heap đổi dấu. (2) `heapreplace`. (3) Dừng khi `reduced ≥ total/2`.

**Bài 5.** (1) Mỗi số dùng tối đa một lần ⇒ lợi ích `v/2` cố định. (2) Chọn các số lớn nhất trước. (3) Cộng dồn tới khi đạt ngưỡng (chứng minh: tập `k` số lớn nhất có tổng lớn nhất).

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
assert Solution().halveArray([1, 1, 1, 1]) == 4
print("Bài 1:", Solution().halveArray([1, 1, 1, 1]), "và", Solution().halveArray([100]), "| vét cạn:", halve_brute([1, 1, 1, 1]), halve_brute([100]))
assert Solution().halveArray([1, 1, 1, 1]) == halve_brute([1, 1, 1, 1])
assert Solution().halveArray([2, 3, 10]) == 2
trace_heap([2, 3, 10])

Bài 1: 4 và 1 | vét cạn: 4 1
tổng = 15, cần giảm >= 7.5
bước 1: chia đôi 10.0 -> 5.0, reduced = 5.0, mảng = [5.0, 3.0, 2.0]
bước 2: chia đôi 5.0 -> 2.5, reduced = 7.5, mảng = [3.0, 2.5, 2.0]


2

In [35]:
# Bài 3, 4
def ops_upper_bound_ans(nums):
    return len(nums)

my_halve_array_ans = halve_heap
rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(1, 10**4) for _ in range(rng.randint(1, 60))]
    r = my_halve_array_ans(nums)
    assert 1 <= r <= ops_upper_bound_ans(nums)
print("Bài 3, 4: OK (đáp án luôn trong [1, n])")

Bài 3, 4: OK (đáp án luôn trong [1, n])


In [36]:
# Bài 5
def halve_once_each_ans(nums):
    target = Fraction(sum(nums), 2); red = Fraction(0); ops = 0
    for v in sorted(nums, reverse=True):
        if red >= target: break
        red += Fraction(v, 2); ops += 1
    return ops

def halve_once_brute(nums):
    n = len(nums); total = sum(nums)
    return min(bin(m).count("1") for m in range(1 << n)
               if 2 * sum(Fraction(nums[i], 2) for i in range(n) if m >> i & 1) >= total)

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 8))]
    assert halve_once_each_ans(nums) == halve_once_brute(nums)
print("Bài 5: OK; [5,19,8,1] ->", halve_once_each_ans([5, 19, 8, 1]), "(so với 3 khi được chia lại)")

Bài 5: OK; [5,19,8,1] -> 4 (so với 3 khi được chia lại)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tham lam lặp trên phần tử lớn nhất":**

```python
h = [-x for x in nums]; heapq.heapify(h)
target = sum(nums) / 2; reduced = ops = 0
while reduced < target:
    x = -h[0]
    heapq.heapreplace(h, -x / 2)    # đưa phần đã giảm trở lại
    reduced += x / 2; ops += 1
return ops
```

In [37]:
def template(nums):
    h = [-float(x) for x in nums]; heapq.heapify(h)
    target = sum(nums) / 2; reduced, ops = 0.0, 0
    while reduced < target:
        x = -h[0]; heapq.heapreplace(h, -x / 2); reduced += x / 2; ops += 1
    return ops
print([template(a) for a, _ in examples])

[3, 3]


**Checklist:**

- [ ] Chia **số thực** `x / 2` (không `x // 2`).
- [ ] Max-heap bằng **đổi dấu**.
- [ ] Đưa nửa trở lại heap (`heapreplace`), vì số đã giảm có thể được chọn lại.
- [ ] Ngưỡng là `sum / 2` (không `sum // 2`), điều kiện dừng `reduced ≥ target`.
- [ ] Đáp án ≤ `n` (dùng để ước lượng thời gian).
- [ ] Đã kiểm tham lam bằng vét cạn chính xác trên mảng nhỏ.